# Decision Tree สำหรับทำนายความเสี่ยงการเสียชีวิตของผู้ป่วย COVID-19

Notebook นี้สร้างและประเมินโมเดล **Decision Tree** เพื่อทำนาย `DEATH_STATUS`

- `0` = Survived
- `1` = Died

จุดประสงค์คือพัฒนาโมเดลของสมาชิกคนที่ 1 และบันทึกผลในรูปแบบเดียวกับ SVM เพื่อใช้สร้าง `model_comparison.ipynb` ภายหลัง

> โปรเจกต์นี้จัดทำเพื่อการศึกษา ไม่ใช่เครื่องมือวินิจฉัยหรือใช้ตัดสินใจทางการแพทย์

## ขั้นตอนการทดลอง

1. ตรวจสอบข้อมูลและคุณภาพข้อมูล
2. สำรวจสัดส่วน Target, Missing values, Duplicate rows และอายุ
3. ใช้ preprocessing และ train/test split ส่วนกลาง
4. ปรับพารามิเตอร์ด้วย Stratified 5-Fold Cross-Validation บน training set
5. เทรนโมเดลที่ดีที่สุดและประเมิน test set เพียงครั้งเดียว
6. แสดง Confusion Matrix, ROC Curve, Precision-Recall Curve, ต้นไม้ และ Feature Importance
7. บันทึก metrics และกราฟใน `results/`

เกณฑ์หลักในการเลือกพารามิเตอร์คือ **F1-score ของคลาส Died** เพราะข้อมูลมีคลาสไม่สมดุล

In [ ]:
from pathlib import Path
import sys
import time
import warnings

# รองรับการรันจากโฟลเดอร์หลักและจากโฟลเดอร์ notebooks
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ใช้เฉพาะกรณี Python หลักไม่มี packaging แต่ .venv มีอยู่
try:
    import packaging  # noqa: F401
except ModuleNotFoundError:
    venv_site_packages = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
    if venv_site_packages.exists():
        sys.path.append(str(venv_site_packages))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

from src.evaluate import evaluate_classifier, print_metrics, save_metrics
from src.preprocess import (
    CATEGORICAL_FEATURES,
    DATA_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    build_preprocessor,
    get_train_test_data,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures" / "decision_tree"
METRICS_PATH = RESULTS_DIR / "decision_tree_metrics.json"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename):
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    print(f"Saved figure: {output_path.relative_to(PROJECT_ROOT)}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 1. ตรวจสอบข้อมูลดิบ

ตารางนี้ใช้ตรวจจำนวนแถว คอลัมน์ Missing values และข้อมูลซ้ำก่อน preprocessing

**การตัดสินใจเรื่อง Duplicate:** Notebook จะแสดงจำนวนข้อมูลซ้ำแต่ยังไม่ลบทิ้ง เพราะ Dataset ไม่มีรหัสผู้ป่วยที่ใช้ยืนยันว่าเป็นบุคคลเดียวกัน การลบอาจทำให้ผู้ป่วยคนละคนที่มีคุณลักษณะเหมือนกันหายไป ทั้ง Decision Tree และ SVM จึงใช้ข้อมูลชุดเดียวกันตาม `src/preprocess.py`

In [ ]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)

dataset_audit = pd.DataFrame(
    {
        "รายการ": [
            "จำนวนแถว",
            "จำนวนคอลัมน์ทั้งหมด",
            "จำนวน features ที่ใช้",
            "จำนวนแถวซ้ำทั้งแถว",
            "Target ที่หาย",
        ],
        "ค่า": [
            len(raw_df),
            raw_df.shape[1],
            len(FEATURE_COLUMNS),
            int(raw_df.duplicated().sum()),
            int(raw_df[TARGET_COLUMN].isna().sum()),
        ],
    }
)

display(dataset_audit)
display(raw_df[FEATURE_COLUMNS + [TARGET_COLUMN]].head())

## 2. การกระจายของ Target

หากทำนายทุกคนเป็น `Survived` จะได้ Accuracy สูงเพราะข้อมูลไม่สมดุล แต่ Recall และ F1 ของ `Died` จะเป็นศูนย์ จึงต้องใช้ Precision, Recall, F1-score และ PR-AUC ร่วมกัน

In [ ]:
target_summary = (
    raw_df[TARGET_COLUMN]
    .value_counts()
    .rename_axis("DEATH_STATUS")
    .reset_index(name="Count")
)
target_summary["Percent"] = target_summary["Count"] / len(raw_df) * 100
display(target_summary.style.format({"Percent": "{:.2f}%"}))

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(
    data=target_summary,
    x="DEATH_STATUS",
    y="Count",
    hue="DEATH_STATUS",
    palette={"Survived": "#4C78A8", "Died": "#E45756"},
    legend=False,
    ax=ax,
)
ax.set_title("Target Distribution")
ax.set_xlabel("")
ax.set_ylabel("Patients")
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}")
fig.tight_layout()
save_figure(fig, "01_target_distribution.png")
plt.show()

## 3. Missing values และรหัส 97, 98, 99

ใน Dataset นี้ค่า `97`, `98` และ `99` ของตัวแปรหมวดหมู่มักหมายถึงไม่ทราบหรือไม่เกี่ยวข้อง จึงแปลงเป็น Missing ก่อนแสดงรายงาน เช่น `PREGNANT` อาจเป็นไม่เกี่ยวข้องสำหรับผู้ป่วยชาย

โมเดลใช้ `SimpleImputer(strategy="most_frequent")` จาก preprocessing ส่วนกลาง เพื่อให้ทั้งสองโมเดลอยู่ภายใต้เงื่อนไขเดียวกัน

In [ ]:
audit_features = raw_df[FEATURE_COLUMNS].copy()

for column in FEATURE_COLUMNS:
    audit_features[column] = pd.to_numeric(
        audit_features[column], errors="coerce"
    )

for column in CATEGORICAL_FEATURES:
    audit_features[column] = audit_features[column].replace(
        [97, 98, 99], np.nan
    )

missing_summary = pd.DataFrame(
    {
        "Feature": FEATURE_COLUMNS,
        "Missing Count": [audit_features[c].isna().sum() for c in FEATURE_COLUMNS],
    }
)
missing_summary["Missing Percent"] = (
    missing_summary["Missing Count"] / len(audit_features) * 100
)
missing_summary = missing_summary.sort_values(
    "Missing Percent", ascending=False
).reset_index(drop=True)

display(missing_summary.style.format({"Missing Percent": "{:.2f}%"}))

missing_for_plot = missing_summary[missing_summary["Missing Count"] > 0]
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(
    data=missing_for_plot,
    x="Missing Percent",
    y="Feature",
    color="#F2CF5B",
    ax=ax,
)
ax.set_title("Missing Values After Converting Codes 97, 98 and 99")
ax.set_xlabel("Missing (%)")
ax.set_ylabel("")
fig.tight_layout()
save_figure(fig, "02_missing_values.png")
plt.show()

## 4. อายุและความเสี่ยงการเสียชีวิต

ส่วนนี้แสดงการกระจายอายุแยกตามผลลัพธ์ และคำนวณอัตราการเสียชีวิตตามช่วงอายุ การวิเคราะห์นี้เป็นความสัมพันธ์ในข้อมูล ไม่ใช่ข้อสรุปเชิงสาเหตุทางการแพทย์

In [ ]:
age_summary = (
    raw_df.groupby(TARGET_COLUMN)["AGE"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)
display(age_summary)

age_bins = [0, 17, 29, 39, 49, 59, 69, 79, 200]
age_labels = ["0-17", "18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]

age_analysis = raw_df[["AGE", TARGET_COLUMN]].copy()
age_analysis["AGE_GROUP"] = pd.cut(
    age_analysis["AGE"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
)
age_analysis["DIED"] = (age_analysis[TARGET_COLUMN] == "Died").astype(int)

age_group_summary = (
    age_analysis.groupby("AGE_GROUP", observed=False)
    .agg(Patients=("DIED", "size"), Died=("DIED", "sum"), Death_Rate=("DIED", "mean"))
    .reset_index()
)
age_group_summary["Death_Rate"] *= 100
display(age_group_summary.style.format({"Death_Rate": "{:.2f}%"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    data=raw_df,
    x="AGE",
    hue=TARGET_COLUMN,
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    palette={"Survived": "#4C78A8", "Died": "#E45756"},
    ax=axes[0],
)
axes[0].set_title("Age Distribution by Outcome")
axes[0].set_xlabel("Age")

sns.barplot(
    data=age_group_summary,
    x="AGE_GROUP",
    y="Death_Rate",
    color="#E45756",
    ax=axes[1],
)
axes[1].set_title("Death Rate by Age Group")
axes[1].set_xlabel("Age group")
axes[1].set_ylabel("Death rate (%)")
axes[1].tick_params(axis="x", rotation=35)

fig.tight_layout()
save_figure(fig, "03_age_analysis.png")
plt.show()

## 5. โหลด Train/Test จากส่วนกลาง

`get_train_test_data()` กำหนดเงื่อนไขร่วมกันไว้ดังนี้:

- `test_size=0.20`
- `random_state=42`
- `stratify=y`
- Features 18 ตัว
- ไม่ใช้คอลัมน์ที่ทำให้เกิด Data Leakage เช่น `DATE_DIED`, `DEATH_YEAR`, `RECOVERY_STATUS` และ `RISK_CATEGORY`

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data()

split_summary = pd.DataFrame(
    {
        "Dataset": ["Train", "Test"],
        "Rows": [len(X_train), len(X_test)],
        "Survived": [(y_train == 0).sum(), (y_test == 0).sum()],
        "Died": [(y_train == 1).sum(), (y_test == 1).sum()],
        "Died Percent": [y_train.mean() * 100, y_test.mean() * 100],
    }
)
display(split_summary.style.format({"Died Percent": "{:.2f}%"}))
print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

## 6. สร้าง Pipeline และปรับพารามิเตอร์

Pipeline ทำให้ preprocessing ถูก `fit` จาก training fold เท่านั้น จึงป้องกัน Data Leakage ระหว่าง Cross-Validation

พารามิเตอร์ที่ทดลอง:

- `criterion`: วิธีวัดคุณภาพของจุดแบ่ง
- `max_depth`: จำกัดความลึกเพื่อควบคุม Overfitting
- `min_samples_split`: จำนวนตัวอย่างขั้นต่ำก่อนแบ่งโหนด
- `min_samples_leaf`: จำนวนตัวอย่างขั้นต่ำใน Leaf
- `ccp_alpha`: Cost Complexity Pruning

In [ ]:
decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "classifier",
            DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

parameter_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 5, 7, 10],
    "classifier__min_samples_split": [2, 10],
    "classifier__min_samples_leaf": [5, 10, 20],
    "classifier__ccp_alpha": [0.0, 0.001],
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}

grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="f1",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)

search_start = time.perf_counter()
grid_search.fit(X_train, y_train)
search_seconds = time.perf_counter() - search_start

print(f"Grid search completed in {search_seconds:.2f} seconds")
print(f"Best CV F1: {grid_search.best_score_:.4f}")
print("Best parameters:")
for name, value in grid_search.best_params_.items():
    print(f"  {name.replace('classifier__', '')}: {value}")

## 7. ผล Cross-Validation

ตารางแสดง 10 ชุดพารามิเตอร์ที่มีค่า Mean CV F1 สูงที่สุด ส่วนกราฟแสดงคะแนนที่ดีที่สุดในแต่ละความลึก

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

cv_columns = [
    "param_classifier__criterion",
    "param_classifier__max_depth",
    "param_classifier__min_samples_split",
    "param_classifier__min_samples_leaf",
    "param_classifier__ccp_alpha",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_recall",
    "mean_test_precision",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
]

top_cv_results = (
    cv_results[cv_columns]
    .sort_values("mean_test_f1", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(top_cv_results.style.format(precision=4))

depth_summary = (
    cv_results.groupby("param_classifier__max_depth", as_index=False)
    .agg(Best_CV_F1=("mean_test_f1", "max"))
    .sort_values("param_classifier__max_depth")
)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.lineplot(
    data=depth_summary,
    x="param_classifier__max_depth",
    y="Best_CV_F1",
    marker="o",
    linewidth=2.5,
    color="#4C78A8",
    ax=ax,
)
ax.set_title("Best Cross-Validated F1 by Tree Depth")
ax.set_xlabel("Max depth")
ax.set_ylabel("Best mean CV F1")
ax.set_xticks(depth_summary["param_classifier__max_depth"])
fig.tight_layout()
save_figure(fig, "04_cv_f1_by_depth.png")
plt.show()

## 8. ประเมินโมเดลที่ดีที่สุดบน Test Set

Test set ถูกใช้หลังจากเลือกพารามิเตอร์เสร็จแล้วเท่านั้น นอกจากนี้ยังสร้าง Dummy Baseline ที่ทำนายทุกคนเป็น `Survived` เพื่อแสดงว่า Accuracy สูงเพียงอย่างเดียวไม่เพียงพอ

In [ ]:
best_decision_tree = grid_search.best_estimator_

train_pred = best_decision_tree.predict(X_train)
train_score = best_decision_tree.predict_proba(X_train)[:, 1]
test_pred = best_decision_tree.predict(X_test)
test_score = best_decision_tree.predict_proba(X_test)[:, 1]

train_metrics = evaluate_classifier(y_train, train_pred, train_score)
test_metrics = evaluate_classifier(y_test, test_pred, test_score)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
dummy_pred = dummy.predict(np.zeros((len(y_test), 1)))
dummy_score = np.zeros(len(y_test))
dummy_metrics = evaluate_classifier(y_test, dummy_pred, dummy_score)

print_metrics(test_metrics)

metric_names = ["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]
comparison_table = pd.DataFrame(
    {
        "Metric": metric_names,
        "Decision Tree": [test_metrics[m] for m in metric_names],
        "Dummy Baseline": [dummy_metrics[m] for m in metric_names],
    }
)
display(comparison_table.style.format({"Decision Tree": "{:.4f}", "Dummy Baseline": "{:.4f}"}))

## 9. ตรวจ Overfitting จาก Train/Test Gap

หากคะแนน Train สูงกว่า Test มาก แสดงว่าโมเดลอาจ Overfit แม้จะใช้ Pruning แล้วก็ตาม

In [ ]:
generalization_table = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"],
        "Train": [
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1_score"],
            train_metrics["roc_auc"],
            train_metrics["pr_auc"],
        ],
        "Test": [
            test_metrics["accuracy"],
            test_metrics["precision"],
            test_metrics["recall"],
            test_metrics["f1_score"],
            test_metrics["roc_auc"],
            test_metrics["pr_auc"],
        ],
    }
)
generalization_table["Gap"] = generalization_table["Train"] - generalization_table["Test"]
display(generalization_table.style.format({"Train": "{:.4f}", "Test": "{:.4f}", "Gap": "{:+.4f}"}))

classification_report_table = pd.DataFrame(test_metrics["classification_report"]).T
display(classification_report_table.style.format(precision=4))

## 10. Confusion Matrix

ให้พิจารณา **False Negative** เป็นพิเศษ เพราะหมายถึงผู้ป่วยที่เสียชีวิตจริง แต่โมเดลทำนายว่า Survived

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    labels=[0, 1],
    display_labels=["Survived", "Died"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Decision Tree Confusion Matrix")
fig.tight_layout()
save_figure(fig, "05_confusion_matrix.png")
plt.show()

## 11. ROC Curve และ Precision-Recall Curve

สำหรับข้อมูลที่คลาส `Died` มีจำนวนน้อย ให้ความสำคัญกับ Precision-Recall Curve และ PR-AUC มากเป็นพิเศษ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

roc_tree = RocCurveDisplay.from_predictions(
    y_test,
    test_score,
    name="Decision Tree",
    ax=axes[0],
)
roc_tree.line_.set(color="#4C78A8")

roc_dummy = RocCurveDisplay.from_predictions(
    y_test,
    dummy_score,
    name="Dummy Baseline",
    ax=axes[0],
)
roc_dummy.line_.set(color="#999999", linestyle="--")
axes[0].set_title("ROC Curve")

pr_tree = PrecisionRecallDisplay.from_predictions(
    y_test,
    test_score,
    name="Decision Tree",
    ax=axes[1],
)
pr_tree.line_.set(color="#E45756")

pr_dummy = PrecisionRecallDisplay.from_predictions(
    y_test,
    dummy_score,
    name="Dummy Baseline",
    ax=axes[1],
)
pr_dummy.line_.set(color="#999999", linestyle="--")
axes[1].set_title("Precision-Recall Curve")

fig.tight_layout()
save_figure(fig, "06_roc_pr_curves.png")
plt.show()

## 12. แสดงต้นไม้การตัดสินใจ

แสดงเพียง 3 ระดับแรกเพื่อให้อ่านได้ง่าย ภาพนี้ใช้ช่วยอธิบายกฎการแบ่งข้อมูล แต่ไม่แสดงโครงสร้างทั้งหมดของโมเดล

In [ ]:
fitted_preprocessor = best_decision_tree.named_steps["preprocessor"]
fitted_classifier = best_decision_tree.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

readable_feature_names = [
    name.replace("numeric__", "").replace("categorical__", "")
    for name in feature_names
]

fig, ax = plt.subplots(figsize=(24, 11))
plot_tree(
    fitted_classifier,
    feature_names=readable_feature_names,
    class_names=["Survived", "Died"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    proportion=True,
    ax=ax,
)
ax.set_title("Decision Tree: First 3 Levels", fontsize=16)
fig.tight_layout()
save_figure(fig, "07_decision_tree_first_3_levels.png")
plt.show()

print("Decision rules (first 3 levels):")
print(
    export_text(
        fitted_classifier,
        feature_names=readable_feature_names,
        max_depth=3,
    )
)

## 13. Feature Importance

Feature Importance แสดงว่า feature ใดถูกใช้ลดความไม่บริสุทธิ์ของโหนดมากที่สุด แต่ไม่ได้แปลว่า feature นั้นเป็นสาเหตุของการเสียชีวิต

In [ ]:
feature_importance = (
    pd.DataFrame(
        {
            "Feature": readable_feature_names,
            "Importance": fitted_classifier.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

top_features = feature_importance.head(15)
display(top_features.style.format({"Importance": "{:.4f}"}))

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature",
    color="#59A14F",
    ax=ax,
)
ax.set_title("Top 15 Decision Tree Feature Importances")
ax.set_xlabel("Importance")
ax.set_ylabel("")
fig.tight_layout()
save_figure(fig, "08_feature_importance.png")
plt.show()

## 14. บันทึกผลการทดลอง

ไฟล์ `results/decision_tree_metrics.json` เก็บผลที่ `model_comparison.ipynb` จะนำไปเปรียบเทียบกับ SVM ภายหลัง

In [ ]:
best_index = grid_search.best_index_
best_params = {
    name.replace("classifier__", ""): value
    for name, value in grid_search.best_params_.items()
}

final_metrics = dict(test_metrics)
final_metrics.update(
    {
        "model_name": "Decision Tree",
        "selection_metric": "F1-score of Died class",
        "best_params": best_params,
        "cv_best_f1": float(grid_search.best_score_),
        "cv_best_pr_auc": float(grid_search.cv_results_["mean_test_pr_auc"][best_index]),
        "cv_best_recall": float(grid_search.cv_results_["mean_test_recall"][best_index]),
        "grid_search_seconds": float(search_seconds),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "input_feature_count": int(len(FEATURE_COLUMNS)),
        "encoded_feature_count": int(len(feature_names)),
        "duplicate_rows_audit": int(raw_df.duplicated().sum()),
        "dummy_baseline": {
            "accuracy": dummy_metrics["accuracy"],
            "precision": dummy_metrics["precision"],
            "recall": dummy_metrics["recall"],
            "f1_score": dummy_metrics["f1_score"],
            "roc_auc": dummy_metrics["roc_auc"],
            "pr_auc": dummy_metrics["pr_auc"],
        },
    }
)

save_metrics(final_metrics, METRICS_PATH)
print(f"Saved metrics: {METRICS_PATH.relative_to(PROJECT_ROOT)}")

## สรุป

การสรุปว่าโมเดลใดดีกว่าจะทำหลังจาก SVM พร้อมแล้ว โดยใช้ test set เดียวกันและพิจารณาตามลำดับ:

1. F1-score ของคลาส Died
2. PR-AUC
3. Recall ของคลาส Died
4. Precision
5. Accuracy ใช้ประกอบเท่านั้น

ขั้นตอนถัดไปคือเปิด Pull Request ของ Decision Tree และรอผลจาก `notebooks/svm.ipynb` ก่อนสร้าง `model_comparison.ipynb`